# Sambandh Setu — ML Progress & Synthetic Journey Analysis

**Purpose:** Code record for the project progress report.

This notebook collects the reproducible analysis steps used for:
1. Loading and inspecting the dating-app dataset.
2. Exploring `match_outcome`.
3. Running the initial reduced-feature multiclass Logistic Regression experiment.
4. Inspecting the synthetic Sambandh journey fields.
5. Calculating stage-transition rates, progression scores, and stage-order inconsistencies.

**Important notes**
- The journey fields in the provided dataset are synthetic, not real Sambandh user records.
- The initial multiclass experiment performed poorly; this notebook records it rather than trying many replacement classifiers.
- A progression score or observed stage-transition rate is **not** the same as a personalized probability of forming a mutual relationship.
- The prior binary experiment was reported as using a separate 5,000-row, 6-column synthetic dataset. Its exact feature names, target definition, and training code were not present in the information available while preparing this notebook. A clearly marked template is included at the end rather than inventing those details.


## 1. Install/import libraries

In [ ]:
# If needed in Google Colab, uncomment:
# !pip -q install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 2. Load the main dataset

In [ ]:
# Option A: Upload the CSV from your computer in Google Colab.
from google.colab import files
uploaded = files.upload()

# Select the uploaded CSV by filename.
# Expected file: dating_app_behavior_sambandh_setu_synthetic.csv
csv_path = "dating_app_behavior_sambandh_setu_synthetic.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Dataset structure and missing values

In [ ]:
print("All columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())


## 4. Outcome distribution and basic EDA

In [ ]:
print("OUTCOME DISTRIBUTION")
display(df["match_outcome"].value_counts(dropna=False).to_frame("count"))

outcome_pct = (
    df["match_outcome"].value_counts(normalize=True, dropna=False)
    .mul(100).round(2).to_frame("percentage")
)
display(outcome_pct)

plt.figure(figsize=(11, 5))
order = df["match_outcome"].value_counts().index
sns.countplot(data=df, x="match_outcome", order=order)
plt.title("Distribution of match_outcome")
plt.xlabel("Outcome")
plt.ylabel("Record count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 5. Initial multiclass classifier (reduced starting features)

This reproduces the initial experiment using the six starting features that were recorded in the project notes. It uses a stratified 80/20 split. The reported result from the experiment was about **9.70% accuracy**, compared with a **10.22% majority-class baseline**. Small differences may occur if the exact split, library versions, or preprocessing differ from the original run.

In [ ]:
target_col = "match_outcome"
feature_cols = [
    "relationship_intent",
    "app_usage_time_min",
    "likes_received",
    "mutual_matches",
    "message_sent_count",
    "last_active_hour"
]

model_df = df[feature_cols + [target_col]].dropna(subset=[target_col]).copy()
X = model_df[feature_cols]
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

multiclass_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])

multiclass_model.fit(X_train, y_train)
y_pred = multiclass_model.predict(X_test)

model_accuracy = accuracy_score(y_test, y_pred)
majority_baseline = y_test.value_counts(normalize=True).max()
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Majority-class baseline accuracy: {majority_baseline:.4f}")
print(f"Model accuracy:                  {model_accuracy:.4f}")
print(f"Macro F1-score:                  {macro_f1:.4f}")
print(f"Weighted F1-score:               {weighted_f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


## 6. Confusion matrix for the multiclass experiment

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.title("Confusion Matrix — Reduced-Feature Logistic Regression")
plt.xlabel("Predicted outcome")
plt.ylabel("Actual outcome")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Identify journey and engagement columns

In [ ]:
print("Outcome-related columns:")
print([
    col for col in df.columns
    if any(word in col.lower()
           for word in ["outcome", "relationship", "mutual", "final"])
])

print("\nEngagement-related columns:")
print([
    col for col in df.columns
    if any(word in col.lower()
           for word in ["click", "activity", "message", "engagement"])
])


## 8. Check the duplicate readiness field

In [ ]:
duplicate_a = "synthetic_mutual_readiness_completed"
duplicate_b = "synthetic_mutual_readiness_completed.1"

if duplicate_a in df.columns and duplicate_b in df.columns:
    print("Duplicate columns identical:", df[duplicate_a].equals(df[duplicate_b]))
else:
    print("One or both duplicate-check columns are missing.")


## 9. Define the ordered journey stages

In [ ]:
stage_cols = [
    "synthetic_connection_established",
    "synthetic_setu_opened",
    "synthetic_setu_explored",
    "synthetic_building_understanding_started",
    "synthetic_building_understanding_completed",
    "synthetic_important_discussions_started",
    "synthetic_important_discussions_completed",
    "synthetic_mutual_readiness_started",
    "synthetic_mutual_readiness_completed",
    "synthetic_next_step_decision_started",
    "synthetic_next_step_decision_completed",
    "synthetic_final_stage_completed"
]

missing_stage_cols = [c for c in stage_cols if c not in df.columns]
if missing_stage_cols:
    raise ValueError(f"Missing expected stage columns: {missing_stage_cols}")

# Treat missing stage values as 0 for this exploratory calculation.
# This assumption should be reviewed before using real data.
stages = df[stage_cols].fillna(0).astype(int)

print("Number of stage columns:", len(stage_cols))
display(stages.head())


## 10. Calculate consecutive-stage progression score

Formula used:

\[
\text{Progression Score} =
\frac{\text{Consecutive completed stages}}{12}\times 100
\]

This is a **journey-completion score**, not a relationship probability.

In [ ]:
def get_progress(row):
    completed = 0
    next_stage = None

    for col in stage_cols:
        if row[col] == 1:
            completed += 1
        else:
            next_stage = col
            break

    if next_stage is None:
        next_stage = "All listed stages completed"

    score = completed / len(stage_cols) * 100
    return pd.Series({
        "stages_completed": completed,
        "progression_score": round(score, 2),
        "next_stage": next_stage
    })

progress = stages.apply(get_progress, axis=1)
df_progress = pd.concat([df, progress], axis=1)

display(
    df_progress[
        ["record_id", "stages_completed", "progression_score", "next_stage"]
    ].head(10)
)


## 11. Flag out-of-order stage indicators

A row is flagged if any later stage is marked 1 after an earlier stage is marked 0. This is a simple consistency check; it does not prove that the record is wrong, because the synthetic generation process may have allowed skipped stages.

In [ ]:
# If a row has a 0 followed later by a 1, its cumulative maximum differs from its current value.
stage_inconsistency = (stages.cummax(axis=1) > stages).any(axis=1)
df_progress["stage_inconsistency_check"] = stage_inconsistency

print("Rows flagged by this check:", int(stage_inconsistency.sum()))
print("Total rows:", len(df_progress))

display(
    df_progress.loc[
        df_progress["stage_inconsistency_check"],
        ["record_id", "stages_completed", "progression_score", "next_stage"]
    ].head(10)
)


## 12. Calculate stage-transition rates

For each transition A → B, the descriptive rate is:

\[
\text{Transition Rate} =
\frac{\text{Records with A = 1 and B = 1}}
{\text{Records with A = 1}}\times 100
\]

The denominator is records with the current stage marked 1. This is an observed synthetic-data transition rate, not a personalized prediction.

In [ ]:
transition_rows = []

for i in range(len(stage_cols) - 1):
    current_stage = stage_cols[i]
    next_stage = stage_cols[i + 1]

    eligible_mask = stages[current_stage] == 1
    eligible = int(eligible_mask.sum())
    progressed = int(
        ((stages[current_stage] == 1) & (stages[next_stage] == 1)).sum()
    )
    dropped = eligible - progressed
    rate = (progressed / eligible * 100) if eligible else np.nan

    transition_rows.append({
        "transition": f"{current_stage} → {next_stage}",
        "eligible_records": eligible,
        "progressed_records": progressed,
        "drop_off_records": dropped,
        "transition_rate_pct": round(rate, 2) if pd.notna(rate) else np.nan
    })

transition_table = pd.DataFrame(transition_rows)
display(transition_table)


## 13. Plot stage-transition rates

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = transition_table.dropna(subset=["transition_rate_pct"])
sns.barplot(
    data=plot_df,
    x="transition_rate_pct",
    y="transition",
)
plt.title("Observed Synthetic Stage-Transition Rates")
plt.xlabel("Transition rate (%)")
plt.ylabel("Stage transition")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


## 14. Optional: compare final-stage flag with outcomes (EDA only)

This cell is optional and is included only as an exploratory check. The `match_outcome` labels were part of the separate dating-app dataset and should not be treated as the target for the Sambandh journey-progression system unless a justified link is established.

In [ ]:
if "synthetic_final_stage_completed" in df.columns and "match_outcome" in df.columns:
    print("Final stage vs match outcome (row-normalized proportions):")
    display(pd.crosstab(
        df["synthetic_final_stage_completed"],
        df["match_outcome"],
        normalize="index"
    ).round(3))


## 15. Binary synthetic classifier — reproducibility template

The project notes report a separate synthetic binary dataset with 5,000 rows and 6 columns, with accuracy 0.72, precision 0.7436, recall 0.8616, and F1-score 0.7983. The exact six-column dataset and target/feature names were not available in the notes used to prepare this notebook. Do not run this template until you load that binary dataset and fill in the correct target and feature names. This avoids inventing the original experiment details.

In [ ]:
# OPTIONAL TEMPLATE — fill these names from the original binary experiment.
#
# binary_df = pd.read_csv("YOUR_BINARY_SYNTHETIC_DATASET.csv")
# print(binary_df.shape)
# print(binary_df.columns.tolist())
# display(binary_df.head())
#
# TARGET = "replace_with_actual_binary_target_column"
# FEATURES = [
#     "replace_with_actual_feature_1",
#     "replace_with_actual_feature_2",
#     # add the actual feature columns used in the original experiment
# ]
#
# X_bin = binary_df[FEATURES]
# y_bin = binary_df[TARGET]
#
# Xb_train, Xb_test, yb_train, yb_test = train_test_split(
#     X_bin, y_bin, test_size=0.20, random_state=42, stratify=y_bin
# )
#
# # Add preprocessing based on the actual feature data types before fitting.
# # Then fit the same classifier used in the original binary experiment
# # and compute accuracy, precision, recall, F1-score, and confusion matrix.


## 16. Summary

- The reduced-feature multiclass classifier did not outperform the majority-class baseline.
- Synthetic binary classification metrics were reported separately and require the original six-column data and target definition to reproduce exactly.
- The journey progression score measures consecutive completed stages.
- The transition table summarizes observed rates in the synthetic data.
- Stage-order inconsistencies and the duplicated readiness field should be cleaned before drawing conclusions.
- Reliable personalized relationship or next-stage probabilities require appropriate real historical outcomes and validation.